# Trois stratégies d'embedding positionnel pour Vision Transformers

| | |
|---|---|
| **Architecture** | Vision Transformer (ViT) |
| **Sujet** | Encodage de la position des patches dans la séquence |
| **Papiers** | [Dosovitskiy et al. 2020 (*ViT*)](https://arxiv.org/abs/2010.11929) · [Vaswani et al. 2017 (*Transformer*)](https://arxiv.org/pdf/1706.03762) · [Su et al. 2021 (*RoPE*)](https://arxiv.org/abs/2104.09864) |
| **Stratégies couvertes** | Absolu appris · Sinusoïdal fixe · Rotatif relatif (RoPE) |
| **Langage** | Python / PyTorch |

---

On part de la question « pourquoi la position compte-t-elle ? » et on
construit chaque stratégie pas à pas. La dernière section compare les
trois variantes sur de petits ViT identiques.

## Imports et configuration

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from vit_from_scratch import (
    CIFAR10_CLASSES,
    CIFAR10_MEAN,
    CIFAR10_STD,
    CosinePositionEmbedding,
    LearnedPositionEmbedding,
    ViTConfig,
    VisionTransformer,
    apply_rope,
    build_dataloaders,
    build_rope_cache,
    build_rope_cache_2d,
    unnormalize_image,
)

torch.manual_seed(7)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

---
## 1. Pourquoi la position compte-t-elle ?

### L'attention est invariante par permutation

Le mécanisme d'auto-attention calcule, pour chaque token $i$, une somme pondérée
de toutes les valeurs :

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

Si l'on permute l'ordre des tokens, les scores changent de ligne, mais
**la sortie globale est simplement permutée de la même façon**, elle ne
contient aucune information sur *où* se trouvait chaque token.
Le modèle traite la séquence comme un **sac de tokens**, sans ordre.

### Pourquoi l'ordre spatial est-il important pour les images ?

Pour le texte, la position encode la syntaxe (sujet avant verbe, etc.).
Pour les images, elle encode la **structure spatiale** :

- Des patches voisins partagent souvent du contenu (même objet, même texture).
- Des patches distants correspondent à des régions sémantiquement différentes.
- Sans position, un patch du coin supérieur gauche et un patch du centre sont traités identiquement.

**Analogie** : imaginez une phrase dont les mots sont mélangés aléatoirement.
Vous retrouvez le vocabulaire, mais perdez complètement la syntaxe et le sens.
C'est exactement ce qui se passe avec une image dont les patches sont brassés.

### Démonstration : permutation et invariance

Sans embedding positionnel, l'attention produit la même sortie
(à permutation près) quel que soit l'ordre des tokens.
Vérifions-le empiriquement.

In [ ]:


# Séquence de 5 tokens, dimension 16
B, N, D = 1, 5, 16
tokens = torch.randn(B, N, D)

# Projection Q, K, V minimale (sans biais, non entraînée)
W_qkv = torch.randn(D, 3 * D) * (D ** -0.5)

def attention_no_pe(x: torch.Tensor) -> torch.Tensor:
    """Auto-attention scalée sans positional embedding."""
    qkv = x @ W_qkv                     # [B, N, 3D]
    q, k, v = qkv.chunk(3, dim=-1)      # chacun [B, N, D]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
    attn = scores.softmax(dim=-1)
    return attn @ v                     # [B, N, D]

# Permutation aléatoire des tokens
perm = torch.randperm(N)
tokens_shuffled = tokens[:, perm, :]

out_original = attention_no_pe(tokens)
out_shuffled  = attention_no_pe(tokens_shuffled)

# La sortie permutée doit correspondre à la sortie originale réordonnée
max_diff = (out_original[:, perm, :] - out_shuffled).abs().max().item()

print(f"Différence max entre out_original[perm] et out_shuffled : {max_diff:.2e}")
print("→ L'attention sans PE est bien invariante par permutation (diff ≈ 0).")

---
## 2. Embedding positionnel absolu appris

### Intuition

L'idée la plus directe : traiter les positions exactement comme des mots.
En NLP, chaque mot possède un vecteur d'embedding appris ; ici, **chaque
position** possède un vecteur d'embedding appris.  
Le modèle dispose d'une **table de correspondance** de taille
$N \times D$ (nombre de tokens × dimension) qu'il optimise par
rétropropagation, exactement comme n'importe quel autre paramètre.

Rien n'est imposé *a priori* : le modèle découvre lui-même quelle
structure spatiale lui est utile pour la tâche cible.

### Équation

Soit $\mathbf{x} \in \mathbb{R}^{N \times D}$ la séquence de tokens de patches
(incluant le class token).  
L'embedding positionnel appris $E_{\text{pos}} \in \mathbb{R}^{N \times D}$
est simplement **ajouté** :

$$
\mathbf{z}_0 = \mathbf{x} + E_{\text{pos}}
$$

- $E_{\text{pos}}$ est un paramètre `nn.Parameter` initialisé avec une distribution tronquée $\mathcal{N}(0,\, 0.02^2)$.
- La forme est $[1, N, D]$ (batch broadcastable).
- Aucune contrainte de structure : chaque position est libre.

In [ ]:


B, N, D = 2, 17, 64   # 16 patches + 1 class token, embed_dim = 64
tokens = torch.randn(B, N, D)

learned_pe = LearnedPositionEmbedding(num_tokens=N, embed_dim=D)
tokens_learned = learned_pe(tokens)

print("=== LearnedPositionEmbedding ===")
print(f"  Forme entrée  : {tuple(tokens.shape)}")
print(f"  Forme sortie  : {tuple(tokens_learned.shape)}")
print(f"  Forme E_pos   : {tuple(learned_pe.position_embedding.shape)}")
print(f"  Nb paramètres : {learned_pe.position_embedding.numel():,}")

delta = (tokens_learned - tokens)
print(f"  Norme de la correction ajoutée : {delta.norm().item():.4f}")

### Visualisation : similarité entre positions

Après apprentissage, les embeddings positionnels proches dans l'espace
2D devraient présenter une **similarité cosinus élevée**.
Ici les poids sont initialisés (non entraînés), donc la structure est
quasi aléatoire, c'est l'état de départ ; l'entraînement fera émerger
des blocs de cohérence spatiale.

La heatmap ci-dessous représente la similarité cosinus entre chaque paire
de vecteurs positionnels $E_{\text{pos}}[i]$ et $E_{\text{pos}}[j]$.

In [ ]:
E = learned_pe.position_embedding.squeeze(0).detach()   # [N, D]
# Similarité cosinus paire-à-paire
E_norm = F.normalize(E, dim=-1)                         # [N, D]
sim = E_norm @ E_norm.T                                 # [N, N]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# — Heatmap de similarité cosinus
im = axes[0].imshow(sim.numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Similarité cosinus entre positions\n(poids initiaux, non entraînés)", fontsize=11)
axes[0].set_xlabel("Position $j$")
axes[0].set_ylabel("Position $i$")
plt.colorbar(im, ax=axes[0])

# — Distribution des valeurs des poids
axes[1].hist(E.numpy().ravel(), bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[1].set_title("Distribution des valeurs de $E_{pos}$\n(init tronquée $\\mathcal{N}(0, 0.02^2)$)", fontsize=11)
axes[1].set_xlabel("Valeur du paramètre")
axes[1].set_ylabel("Effectif")

plt.tight_layout()
plt.show()

### Bilan : embedding absolu appris

| | |
|---|---|
| ✅ **Flexible** | Aucune contrainte de structure ; s'adapte complètement à la tâche et aux données. |
| ✅ **Simple** | Un seul `nn.Parameter` ; implémentation triviale. |
| ❌ **Longueur fixe** | La table a une taille $N$ fixée à l'entraînement. Impossible d'inférer sur des images de résolution différente sans interpolation. |
| ❌ **Pas d'extrapolation** | Si l'on voit une séquence plus longue que $N$, il n'y a pas de position définie. |
| ❌ **Pas de biais inductif spatial** | Le modèle doit *apprendre* que les positions voisines se ressemblent, ce que le sinusoïdal encode gratuitement. |

> **Utilisé dans** : ViT original [(Dosovitskiy et al. 2020)](https://arxiv.org/abs/2010.11929), [BERT](https://arxiv.org/pdf/1810.04805), [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf).

---
## 3. Embedding positionnel sinusoïdal fixe

### Intuition

Au lieu d'apprendre une table, on l'**encode analytiquement** à partir
d'une famille de sinusoïdes à fréquences progressivement plus élevées,
comme une **décomposition de Fourier** de la position.

Chaque dimension $i$ de l'embedding porte une fréquence différente :
- Les **grandes longueurs d'onde** (basses fréquences) codent les
  positions à grande échelle (*token 1 vs token 100*).
- Les **courtes longueurs d'onde** (hautes fréquences) discriminent
  les positions adjacentes (*token 5 vs token 6*).

Les dimensions vont par **paires sin/cos** : ensemble, elles forment
une **rotation dans le plan** à la fréquence $\omega_i$, ce qui rend
le produit scalaire entre deux positions sensible uniquement à
leur **distance**, une propriété très désirable.

### Équations

Pour une position $pos \in \{0, \ldots, N-1\}$ et une paire de
dimensions $(2i,\, 2i+1)$ avec $D$ la dimension totale :

$$
PE(pos,\, 2i) = \sin\!\left(\frac{pos}{10000^{\,2i/D}}\right)
\qquad
PE(pos,\, 2i+1) = \cos\!\left(\frac{pos}{10000^{\,2i/D}}\right)
$$

La **base** $10000$ garantit que les fréquences s'échelonnent sur
plusieurs ordres de grandeur : la fréquence la plus basse a une période
de $2\pi \times 10000$ tokens, bien au-delà de toute séquence pratique.

**Propriété clé** : le produit scalaire
$PE(pos_1) \cdot PE(pos_2)$ dépend uniquement de $|pos_1 - pos_2|$
(c'est la somme de cosinus des différences d'angle), ce qui injecte
un **biais de distance relative** dès le début de l'entraînement.

In [ ]:
# Visualiser les sinusoïdes positionnelles à différentes fréquences

embed_dim_plot = 32
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

def plot_pe_family(ax, positions, pair_indices, title):
    for pair_idx, color in zip(pair_indices, colors):
        angle_rate = 1.0 / (10000 ** (2 * pair_idx / embed_dim_plot))
        angle = positions * angle_rate
        ax.plot(positions, torch.sin(angle).numpy(), color=color, linestyle='-', label=f'sin dim {2 * pair_idx}')
        ax.plot(positions, torch.cos(angle).numpy(), color=color, linestyle='--', label=f'cos dim {2 * pair_idx + 1}')
    ax.set_title(title)
    ax.set_ylabel('Valeur de PE')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)

# 1) Fenêtre courte pour visualiser les fréquences rapides
positions_short = torch.arange(0, 161, dtype=torch.float32)
fast_pairs = [0, 1, 2, 3]

fig, ax = plt.subplots(1, 1, figsize=(18, 5), constrained_layout=True)
plot_pe_family(
    ax,
    positions_short,
    fast_pairs,
    'PE sin/cos - fréquences rapides sur une fenêtre courte (0 à 160)',
 )
ax.set_xlabel('Position')
plt.show()

# 2) Fenêtre longue pour visualiser les fréquences lentes
positions_long = torch.arange(0, 4097, 16, dtype=torch.float32)
slow_pairs = [12, 13, 14, 15]

fig, ax = plt.subplots(1, 1, figsize=(18, 5), constrained_layout=True)
plot_pe_family(
    ax,
    positions_long,
    slow_pairs,
    'PE sin/cos - fréquences lentes sur une fenêtre longue (0 à 4096)',
 )
ax.set_xlabel('Position')
plt.show()

# Panneau séparé : échelonnement des fréquences selon les dimensions
pair_ids = torch.arange(embed_dim_plot // 2, dtype=torch.float32)
angular_rates = 1.0 / (10000 ** (2 * pair_ids / embed_dim_plot))
periods = (2 * math.pi) / angular_rates

fig, ax = plt.subplots(1, 1, figsize=(12, 4), constrained_layout=True)
ax.plot(pair_ids.numpy(), angular_rates.numpy(), marker='o', label='Fréquence angulaire')
ax.set_yscale('log')
ax.set_title('Échelonnement des fréquences selon les dimensions')
ax.set_xlabel('Indice de paire i')
ax.set_ylabel('Fréquence angulaire (échelle log)')
ax.grid(True, alpha=0.3)

ax2 = ax.twinx()
ax2.plot(pair_ids.numpy(), periods.numpy(), color='tab:purple', marker='s', alpha=0.75, label='Période')
ax2.set_yscale('log')
ax2.set_ylabel('Période correspondante')

handles_1, labels_1 = ax.get_legend_handles_labels()
handles_2, labels_2 = ax2.get_legend_handles_labels()
ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc='upper right', fontsize=9)

plt.show()

### Lecture des courbes

**Graphique 1 - fréquences rapides (paires 0-3, fenêtre 0-160).**
Les premières paires oscillent plusieurs fois sur cent positions : chaque position voisine
reçoit un angle sensiblement différent, ce qui donne à chaque token une "empreinte locale"
unique. Deux positions adjacentes (par exemple 50 et 51) ont des valeurs sin/cos distinctes
dans ces dimensions, ce qui permet au modèle de les distinguer facilement.

**Graphique 2 - fréquences lentes (paires 12-15, fenêtre 0-4096).**
Les dernières paires complètent à peine une oscillation sur toute la fenêtre : elles
capturent la position globale dans la séquence (debut, milieu, fin). Un token en position
100 et un token en position 3000 diffèrent fortement sur ces dimensions, même si leurs
voisins immédiats se ressemblent.

**Graphique 3 - spectre des fréquences (échelle log).**
La fréquence angulaire décroit de façon log-linéaire avec l'indice de paire : on couvre
plusieurs ordres de grandeur, des courtes longueurs d'onde (discrimination locale) aux
longues longueurs d'onde (structure globale). La courbe de période confirme que la
dernière paire a une période de l'ordre de $2\pi \times 10000 \approx 62\,800$ positions,
bien au-delà de toute séquence pratique.

**Vue d'ensemble.**
L'empilement de toutes ces paires forme un "repère multi-echelle" : chaque position
possède une combinaison unique de valeurs sin/cos sur l'ensemble du spectre, comme
une empreinte digitale positionnelle. Les courtes longueurs d'onde différencient les
voisins, les longues longueurs d'onde situent le token dans la séquence globale.

In [ ]:


cosine_pe = CosinePositionEmbedding(num_tokens=N, embed_dim=D)
tokens_cosine = cosine_pe(tokens)

print("=== CosinePositionEmbedding ===")
print(f"  Forme entrée  : {tuple(tokens.shape)}")
print(f"  Forme sortie  : {tuple(tokens_cosine.shape)}")
print(f"  Forme buffer  : {tuple(cosine_pe.position_embedding.shape)}")
print(f"  Nb paramètres : 0  (buffer non entraîné)")

# Comparaison de la norme de correction
delta_learned = (tokens_learned - tokens).norm().item()
delta_cosine  = (tokens_cosine  - tokens).norm().item()
print(f"\n  Norme correction learned   : {delta_learned:.4f}")
print(f"  Norme correction sinusoïdal: {delta_cosine:.4f}")

### Visualisation : la matrice sinusoïdale

Affichons la matrice $PE \in \mathbb{R}^{N \times D}$ sous deux angles :

1. **Heatmap**, on voit directement les motifs de fréquence : les
   colonnes de gauche oscillent lentement (basse fréquence), celles
   de droite oscillent rapidement.
2. **Similarité cosinus**, on observe que les positions *proches*
   (diagonale) ont une forte similarité, qui décroît avec la distance.
   C'est le **biais de localité spatiale** encodé gratuitement.

In [ ]:
PE = cosine_pe.position_embedding.squeeze(0).detach()   # [N, D]
PE_norm = F.normalize(PE, dim=-1)
sim_cos = PE_norm @ PE_norm.T

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# — Matrice PE (motifs sinusoïdaux)
im0 = axes[0].imshow(PE.numpy(), aspect="auto", cmap="seismic", vmin=-1, vmax=1)
axes[0].set_title("Matrice $PE$ sinusoïdale\n(lignes = positions, colonnes = dimensions)", fontsize=11)
axes[0].set_xlabel("Dimension $i$")
axes[0].set_ylabel("Position $pos$")
plt.colorbar(im0, ax=axes[0])

# — Similarité cosinus entre positions
im1 = axes[1].imshow(sim_cos.numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Similarité cosinus entre positions\n(biais de localité encodé analytiquement)", fontsize=11)
axes[1].set_xlabel("Position $j$")
axes[1].set_ylabel("Position $i$")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

### Bilan : embedding sinusoïdal fixe

| | |
|---|---|
| ✅ **Aucun paramètre** | La table est déterministe, zéro coût de mémoire supplémentaire lié aux positions. |
| ✅ **Biais inductif spatial** | La similarité cosinus décroît avec la distance dès le début de l'entraînement. |
| ✅ **Extrapolation partielle** | On peut évaluer $PE(pos)$ pour tout $pos$, même hors de la plage d'entraînement. |
| ❌ **Moins flexible** | Structure rigide, pas adaptable à la tâche ou à la distribution des données. |
| ❌ **Encodage absolu** | La représentation dépend de la position absolue, pas de la distance relative entre tokens. |

> **Utilisé dans** : Transformer original [(Vaswani et al. 2017)](https://arxiv.org/pdf/1706.03762), MAE [(He et al. 2022)](https://arxiv.org/abs/2111.06377).

---
## 4. RoPE : embeddings positionnels rotatifs

### Intuition

Les deux stratégies précédentes *ajoutent* un vecteur aux tokens avant
l'attention, elles modifient la séquence d'entrée.
RoPE adopte une philosophie différente :

> **Ne pas modifier les tokens, mais faire tourner les requêtes $q$ et les clés $k$
> juste avant le calcul des scores d'attention.**

L'opération de rotation est choisie de telle sorte que le produit
scalaire $q_i^\top k_j$ après rotation dépend **uniquement de la
position relative** $i - j$, et non des positions absolues $i$ et $j$
séparément. C'est la propriété fondamentale de RoPE.

**Pourquoi les paires sin/cos forment-elles une rotation ?**  
Pour deux scalaires $(u_1, u_2)$ et un angle $\theta$, la rotation
planaire est :

$$
R(\theta) \begin{bmatrix} u_1 \\ u_2 \end{bmatrix}
= \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}
\begin{bmatrix} u_1 \\ u_2 \end{bmatrix}
= \begin{bmatrix} u_1 \cos\theta - u_2 \sin\theta \\ u_1 \sin\theta + u_2 \cos\theta \end{bmatrix}
$$

La rotation **préserve la norme** ($\|R(\theta)u\| = \|u\|$), et
le produit scalaire après rotation :

$$
\bigl(R(m\theta)\, q\bigr)^\top \bigl(R(n\theta)\, k\bigr)
= q^\top R\bigl((n-m)\theta\bigr)\, k
$$

ne dépend que de la **distance** $n - m$, pas de $m$ ni de $n$ pris
séparément.

### Équation complète

On découpe la dimension de tête $d$ en $d/2$ paires. Pour la paire
$(2t, 2t+1)$ à la position $pos$ :

$$
\theta_t = \frac{1}{10000^{\,2t/d}}
$$

$$
\text{RoPE}(x_{2t}, x_{2t+1},\, pos)
= \begin{bmatrix}
    x_{2t}   \cos(pos\,\theta_t) - x_{2t+1} \sin(pos\,\theta_t) \\
    x_{2t}   \sin(pos\,\theta_t) + x_{2t+1} \cos(pos\,\theta_t)
  \end{bmatrix}
$$

En pratique, on précalcule un **cache** $\cos$ et $\sin$ de forme
$[1, 1, N, d]$ (broadcastable sur batch et têtes), puis on applique
la rotation à $q$ et $k$ avant le produit scalaire.

### Extension naturelle : RoPE 2D

Pour une image, la position d'un patch n'est pas un entier mais un couple
$(x, y)$ sur une grille. RoPE 2D généralise l'idée en appliquant deux
rotations indépendantes : une pour l'axe horizontal $x$, une pour l'axe
vertical $y$. Concrètement, on découpe encore la tête en paires, mais on
réserve une moitié des dimensions aux coordonnées $x$ et l'autre moitié
aux coordonnées $y$.

L'effet recherché est le même qu'en 1D : le score d'attention dépend des
**déplacements relatifs** $(\Delta x, \Delta y)$ plutôt que des coordonnées
absolues. Cela colle mieux à la géométrie des images, où deux patches peuvent
être proches horizontalement, verticalement ou en diagonale.

$$
\text{RoPE2D}(q, (x, y)) = \bigl(R_x(x)\, q_x\;\Vert\; R_y(y)\, q_y\bigr)
$$

avec $R_x$ et $R_y$ deux rotations de même principe, appliquées séparément
sur les sous-espaces associés aux axes $x$ et $y$. En pratique, on peut
visualiser RoPE 2D comme une version « grille » de RoPE 1D : une rotation
par axe, puis le produit scalaire reste sensible à la distance relative
sur chaque dimension spatiale.

In [ ]:

B_r, H, N_r, D_head = 2, 4, 17, 16  # batch, têtes, tokens, dim par tête
q = torch.randn(B_r, H, N_r, D_head)
k = torch.randn(B_r, H, N_r, D_head)

cos_cache, sin_cache = build_rope_cache(seq_len=N_r, head_dim=D_head)

q_rope = apply_rope(q, cos_cache, sin_cache)
k_rope = apply_rope(k, cos_cache, sin_cache)

print("=== RoPE ===")
print(f"  Forme q / k              : {tuple(q.shape)}")
print(f"  Forme cache cos / sin    : {tuple(cos_cache.shape)}")
print(f"  Forme q_rope / k_rope    : {tuple(q_rope.shape)}")
print(f"  Nb paramètres RoPE       : 0  (cache déterministe)")

# Vérification : la rotation préserve la norme
norm_q      = q.norm(dim=-1)
norm_q_rope = q_rope.norm(dim=-1)
max_norm_diff = (norm_q - norm_q_rope).abs().max().item()
print(f"\n  Différence max de norme (q vs q_rope) : {max_norm_diff:.2e}")
print("  → RoPE préserve la norme (≈ 0).")

### Vérification : le score dépend de la distance relative

Vérifions empiriquement que $q_i^\top k_j$ après RoPE dépend uniquement
de $i - j$.  On compare le score de la paire $(i=3, j=5)$ avec celui
de la paire $(i=7, j=9)$, même distance $\Delta = 2$, mais positions
absolues différentes.

In [ ]:
# Même vecteur q et k, positions absolues différentes mais distance identique

q_vec = torch.randn(1, 1, 1, D_head)  # un seul vecteur q
k_vec = torch.randn(1, 1, 1, D_head)  # un seul vecteur k

def score_at_distance(pos_q: int, pos_k: int) -> float:
    """Score scalé q^T k après rotation RoPE aux positions pos_q et pos_k."""
    q_seq = torch.zeros(1, 1, N_r, D_head)
    k_seq = torch.zeros(1, 1, N_r, D_head)
    q_seq[0, 0, pos_q] = q_vec[0, 0, 0]
    k_seq[0, 0, pos_k] = k_vec[0, 0, 0]
    q_r = apply_rope(q_seq, cos_cache, sin_cache)
    k_r = apply_rope(k_seq, cos_cache, sin_cache)
    return (q_r[0, 0, pos_q] @ k_r[0, 0, pos_k]).item() / math.sqrt(D_head)

s1 = score_at_distance(3, 5)   # distance 2
s2 = score_at_distance(7, 9)   # distance 2
s3 = score_at_distance(0, 4)   # distance 4

print(f"Score (pos 3 → pos 5, Δ=2) : {s1:.6f}")
print(f"Score (pos 7 → pos 9, Δ=2) : {s2:.6f}")
print(f"→ Même distance, même score : {abs(s1 - s2) < 1e-5}")
print()
print(f"Score (pos 0 → pos 4, Δ=4) : {s3:.6f}")
print(f"→ Distance différente, score différent : {abs(s1 - s3) > 1e-5}")

### RoPE 2D pour les Vision Transformers

#### Pourquoi la position 1D est-elle insuffisante pour les images ?

En NLP, les tokens forment une séquence plate : la position $0, 1, \ldots, N-1$ encode
fidèlement l'ordre du texte. Pour les images, les patches sont distribués sur une **grille
2D** : un patch est repéré par une coordonnée de ligne $r$ et une coordonnée de colonne $c$.
Aplatir cette grille en séquence brise la topologie :

- Le patch 7 (fin de la ligne 0 pour une grille de largeur 8) et le patch 8 (debut de la
  ligne 1) sont **adjacents en 1D** mais séparés par toute la largeur de l'image en 2D.
- RoPE 1D leur attribue une distance de 1 dans les rotations, alors qu'ils ne sont pas voisins.

#### Comment RoPE 2D encode-t-il les deux axes ?

RoPE 2D étend le principe rotatoire en découpant la dimension de tete $d$ en **deux moitiés égales** :

- La **première moitié** (dimensions $0, \ldots, d/2 - 1$) reçoit la rotation associée à la
  coordonnée de **ligne** $r$.
- La **seconde moitié** (dimensions $d/2, \ldots, d - 1$) reçoit la rotation associée à la
  coordonnée de **colonne** $c$.

Formellement, soit $q$ un vecteur requete de dimension $d$ et $(r, c)$ les coordonnées
du patch correspondant. On note $q_r$ et $q_c$ les deux sous-vecteurs de dimension $d/2$ :

$$
q' = \bigl(R_r(r) \cdot q_r\bigr) \;\|\; \bigl(R_c(c) \cdot q_c\bigr)
$$

La même transformation s'applique aux clés $k$. La rotation $R_r$ utilise les fréquences
$\theta_t = 10000^{-2t/(d/2)}$ appliquées à la coordonnée de ligne, et $R_c$ les mêmes
fréquences appliquées à la coordonnée de colonne.

#### Le score dépend des déplacements relatifs

Le score d'attention entre deux patches de positions $(r_1, c_1)$ et $(r_2, c_2)$ devient :

$$
q'^{\,\top} k' \;=\; f\!\left(\Delta r,\, \Delta c\right), \qquad \text{avec}\; \Delta r = r_2 - r_1,\; \Delta c = c_2 - c_1
$$

Ce score **ne dépend que des déplacements relatifs** $(\Delta r, \Delta c)$, pas des
coordonnées absolues. Deux paires de patches séparées du même vecteur $(+1, +2)$ sur la
grille obtiennent exactement le même score d'attention, quelle que soit leur position dans
l'image. C'est un biais de position relative en 2D, entièrement déterministe, sans aucun
paramètre appris.

Cette approche est implémentée dans `build_rope_cache_2d` de la bibliothèque `vit_from_scratch`.

In [ ]:
# Visualiser RoPE 1D et son extension 2D sur des scores de similarité relatifs
def rope_inv_freq(dim: int) -> torch.Tensor:
    pair_ids = torch.arange(dim // 2, dtype=torch.float32)
    return 1.0 / (10000 ** (2 * pair_ids / dim))


def rotate_rope_1d(vec: torch.Tensor, position: float, inv_freq: torch.Tensor) -> torch.Tensor:
    angles = position * inv_freq
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    even = vec[0::2]
    odd = vec[1::2]
    rotated = torch.empty_like(vec)
    rotated[0::2] = even * cos - odd * sin
    rotated[1::2] = even * sin + odd * cos
    return rotated


def score_rope_1d(pos_q: int, pos_k: int, q_base: torch.Tensor, k_base: torch.Tensor, inv_freq: torch.Tensor) -> float:
    q_rot = rotate_rope_1d(q_base, float(pos_q), inv_freq)
    k_rot = rotate_rope_1d(k_base, float(pos_k), inv_freq)
    return float((q_rot @ k_rot).item() / math.sqrt(q_base.numel()))


def rotate_rope_2d(vec: torch.Tensor, pos_xy: tuple[int, int], inv_freq: torch.Tensor) -> torch.Tensor:
    half = vec.numel() // 2
    x_pos, y_pos = pos_xy
    x_part = rotate_rope_1d(vec[:half], float(x_pos), inv_freq)
    y_part = rotate_rope_1d(vec[half:], float(y_pos), inv_freq)
    return torch.cat([x_part, y_part])


def score_rope_2d(
    pos_q: tuple[int, int],
    pos_k: tuple[int, int],
    q_base: torch.Tensor,
    k_base: torch.Tensor,
    inv_freq: torch.Tensor,
) -> float:
    q_rot = rotate_rope_2d(q_base, pos_q, inv_freq)
    k_rot = rotate_rope_2d(k_base, pos_k, inv_freq)
    return float((q_rot @ k_rot).item() / math.sqrt(q_base.numel()))



D_vis = 32
q_base_1d = torch.randn(D_vis)
k_base_1d = torch.randn(D_vis)
inv_freq_1d = rope_inv_freq(D_vis)

positions_1d = torch.arange(0, 48)
rope_1d_scores = torch.empty(len(positions_1d), len(positions_1d))
for i, pos_q in enumerate(positions_1d):
    for j, pos_k in enumerate(positions_1d):
        rope_1d_scores[i, j] = score_rope_1d(int(pos_q), int(pos_k), q_base_1d, k_base_1d, inv_freq_1d)

grid_h = grid_w = 14
ref_xy = (7, 7)
q_base_2d = torch.randn(D_vis)
k_base_2d = torch.randn(D_vis)
inv_freq_2d = rope_inv_freq(D_vis // 2)

rope_2d_scores = torch.empty(grid_h, grid_w)
for y in range(grid_h):
    for x in range(grid_w):
        rope_2d_scores[y, x] = score_rope_2d(ref_xy, (x, y), q_base_2d, k_base_2d, inv_freq_2d)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)

im0 = axes[0].imshow(rope_1d_scores.numpy(), cmap="RdBu_r", vmin=rope_1d_scores.min().item(), vmax=rope_1d_scores.max().item())
axes[0].set_title("RoPE 1D : score relatif entre positions", fontsize=12)
axes[0].set_xlabel("Position $j$")
axes[0].set_ylabel("Position $i$")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(
    rope_2d_scores.numpy(),
    origin="lower",
    cmap="RdBu_r",
    vmin=rope_2d_scores.min().item(),
    vmax=rope_2d_scores.max().item(),
)
axes[1].scatter([ref_xy[0]], [ref_xy[1]], c="black", s=60, marker="x", label="position de référence")
axes[1].set_title("RoPE 2D : score relatif autour d'une position de référence", fontsize=12)
axes[1].set_xlabel("Coordonnée $x$")
axes[1].set_ylabel("Coordonnée $y$")
axes[1].legend(loc="upper right")
plt.colorbar(im1, ax=axes[1])

plt.show()

### Lecture des visualisations RoPE

**Graphique gauche - RoPE 1D (matrice 48 x 48).**
La matrice des scores présente des bandes diagonales : des positions séparées par la même
distance $|i - j|$ obtiennent des scores similaires. Ce motif confirme empiriquement la
propriété fondamentale de RoPE : le score ne dépend que de la distance relative, pas des
positions absolues. Les bandes oscillent car plusieurs fréquences sont superposées ; la
structure multi-fréquences de RoPE est visible directement dans les scores.

**Graphique droit - RoPE 2D (grille 14 x 14, point de référence au centre).**
Les scores forment des **anneaux concentriques** autour du point de référence : les patches
proches en distance euclidienne 2D obtiennent des scores élevés, les patches éloignés des
scores faibles. Ce motif concentrique est une conséquence directe de la structure des
rotations 2D : les deux axes contribuent indépendamment, et leur combinaison donne une
sensibilité à la norme du déplacement $\sqrt{\Delta r^2 + \Delta c^2}$ en première
approximation.

Ce **biais spatial natif** est injecté sans aucun paramètre appris : dès l'initialisation,
avant tout entraînement, RoPE 2D rend chaque patch nativement attentif à son voisinage
immédiat sur la grille. Le modèle n'a pas besoin de découvrir cette propriété ; elle est
codée dans la structure même des rotations.

**Différence essentielle entre RoPE 1D et RoPE 2D.**
RoPE 1D aplatit les patches en une séquence : le patch 7 (fin de la ligne 0) et le
patch 8 (debut de la ligne 1) sont considérés comme voisins immédiats, même s'ils
sont sur des lignes différentes. RoPE 2D respecte la topologie de la grille : la
proximité est mesurée séparément sur l'axe des lignes et sur l'axe des colonnes,
ce qui donne un biais spatial plus fidèle à la géométrie de l'image. La heatmap circulaire
du graphique droit, absente du graphique gauche, illustre cette différence qualitative.

### Bilan : RoPE

| | |
|---|---|
| ✅ **Position relative** | Le score d'attention dépend uniquement de la distance, biais inductif fort. |
| ✅ **Aucun paramètre** | Cache déterministe, zéro paramètre supplémentaire. |
| ✅ **Norme préservée** | La rotation est une isométrie : pas de distorsion des représentations. |
| ✅ **Extrapolation** | Fonctionne correctement au-delà de la longueur d'entraînement (meilleure que les méthodes additives). |
| ❌ **Seulement $q$ et $k$** | Ne modifie pas les valeurs $v$ ; l'information positionnelle n'est injectée que dans les scores. |
| ❌ **Moins interprétable** | Aucune visualisation directe de « l'embedding de la position $p$ ». |

> **Utilisé dans** : LLaMA [(Touvron et al. 2023)](https://arxiv.org/pdf/2302.13971), Mistral, Qwen, et de nombreux LLMs modernes.
> Pour la vision, des variantes 2D ([RoPE-2D](https://arxiv.org/pdf/2403.13298), EVA-02) étendent la rotation aux axes $(x, y)$.

### Comparaison

$$
\begin{aligned}
\text{Sinusoïdal fixe}\quad &\mathbf{x}_{pos} = \mathbf{x} + PE(pos) \\[4pt]
\text{RoPE 1D}\quad &q' = R(pos)q,\; k' = R(pos)k,\; q'^\top k' = f(\Delta pos) \\[4pt]
\text{RoPE 2D}\quad &q' = R_x(x)R_y(y)q,\; k' = R_x(x)R_y(y)k,\; q'^\top k' = f(\Delta x, \Delta y)
\end{aligned}
$$

En résumé : le sinusoidal fixe ajoute une position absolue, RoPE 1D transforme une position 1D en rotation relative, et RoPE 2D généralise la même idée à une grille d'image en séparant les axes horizontal et vertical.

---
## 5. Tableau comparatif

| Stratégie | Mécanisme | Paramètres | Type de position | Extrapolation | Références |
|-----------|-----------|------------|-----------------|---------------|------------|
| **Appris absolu** | Addition $\mathbf{x} + E_{pos}$ | $N \times D$ appris | Absolue | ✗ | ViT (Dosovitskiy 2020), BERT |
| **Sinusoïdal fixe** | Addition $\mathbf{x} + PE(pos)$ | 0 (buffer) | Absolue | Partielle | Transformer (Vaswani 2017), MAE (He 2022) |
| **RoPE** | Rotation $q, k$ avant attention | 0 (cache) | Relative | Bonne | LLaMA, Mistral, EVA-02 |

**Remarque sur l'application en vision** :  
ViT utilise des positions 1D sur une séquence de patches (avec class token). Les variantes
sinusoïdales 2D (MAE, ViT-MAE) encodent séparément la ligne et la colonne du patch,
ce qui donne un biais spatial plus naturel pour les images.  
RoPE 2D (EVA, InternViT) applique des rotations indépendantes sur les axes $x$ et $y$.

---
## 6. Comparaison sur trois petits Vision Transformers

Instancions trois `VisionTransformer` identiques à l'exception de la
stratégie positionnelle, et vérifions qu'ils produisent des logits de
même forme mais de valeurs différentes, la position est bien
*informatiquement active* dans chaque cas.

In [ ]:

images = torch.randn(2, 3, 32, 32)

common_kwargs = dict(
    image_size=32,
    patch_size=8,
    in_channels=3,
    num_classes=10,
    embed_dim=32,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.0,
    attention_dropout=0.0,
)

configs = {
    "learned": ViTConfig(position_embedding="learned", **common_kwargs),
    "cosine":  ViTConfig(position_embedding="cosine",  **common_kwargs),
    "rope":    ViTConfig(position_embedding="rope",    **common_kwargs),
}

models  = {name: VisionTransformer(cfg) for name, cfg in configs.items()}
logits  = {name: model(images) for name, model in models.items()}
params  = {name: sum(p.numel() for p in model.parameters()) for name, model in models.items()}

print(f"{'Stratégie':<12}  {'Forme logits':<18}  {'Nb paramètres':>15}  {'Moyenne logits':>14}")
print("-" * 65)
for name in ("learned", "cosine", "rope"):
    shape  = tuple(logits[name].shape)
    mean   = logits[name].mean().item()
    print(f"  {name:<10}  {str(shape):<18}  {params[name]:>15,}  {mean:>14.6f}")

### Visualisation : divergence des représentations selon la stratégie

Trois modèles aux poids initialisés de manière identique (même graine), différant uniquement
par leur stratégie positionnelle, produisent déjà des représentations internes distinctes
avant tout entraînement. Cela montre que le choix du PE **n'est pas neutre** : il oriente
le paysage des représentations dès le premier forward pass.

On l'examine sous deux angles complémentaires sur un petit batch d'images CIFAR-10 réelles :

1. **Barplot groupé des logits** : pour une image du batch, on affiche les logits bruts
   des 10 classes CIFAR-10 pour chacune des 3 stratégies. Les barres d'une même classe
   divergent selon la stratégie, ce qui illustre concrètement l'effet positionnel sur la
   prédiction finale.

2. **Projection PCA des tokens CLS** : on extrait le token CLS de chaque modèle via
   `model.encode_tokens(images)[:, 0]`, puis on projette ces vecteurs en 2D par SVD.
   La forme des nuages de points selon la stratégie révèle que les trois modèles organisent
   différemment l'espace des représentations, le PE est la seule variable qui change.

In [ ]:

# ---- Chargement d'un petit batch CIFAR-10 ----
dataloaders = build_dataloaders(
    "cifar10",
    data_dir="../data",
    batch_size=32,
    max_train_samples=200,
    max_val_samples=100,
    download=True,
    image_size=32,
)
cifar_images, cifar_labels = next(iter(dataloaders.train_loader))

# ---- Création de trois ViT compatibles avec common_kwargs ----

cifar_configs = {
    "learned": ViTConfig(position_embedding="learned", **common_kwargs),
    "cosine":  ViTConfig(position_embedding="cosine",  **common_kwargs),
    "rope":    ViTConfig(position_embedding="rope",    **common_kwargs),
}
cifar_models = {name: VisionTransformer(cfg) for name, cfg in cifar_configs.items()}

# ---- Forward pass : logits et tokens CLS ----
pe_names = ["learned", "cosine", "rope"]
pe_colors = ["#4C72B0", "#DD8452", "#55A868"]

cifar_logits = {}
cls_tokens = {}
for name, model in cifar_models.items():
    model.eval()
    with torch.no_grad():
        cifar_logits[name] = model(cifar_images)              # [B, num_classes]
        tokens = model.encode_tokens(cifar_images)            # [B, N+1, embed_dim]
        cls_tokens[name] = tokens[:, 0, :]                   # [B, embed_dim]

# ---- PCA via SVD (sans sklearn) ----
def torch_pca_2d(features: torch.Tensor) -> torch.Tensor:
    """Projette features [N, D] sur ses 2 premiers composants principaux."""
    centered = features - features.mean(dim=0, keepdim=True)
    _, _, Vt = torch.linalg.svd(centered, full_matrices=False)
    return (centered @ Vt[:2].T).numpy()   # [N, 2]

pca_projections = {name: torch_pca_2d(cls_tokens[name]) for name in pe_names}
labels_np = cifar_labels.numpy()
num_cifar_classes = len(CIFAR10_CLASSES)
cmap = plt.get_cmap("tab10")

# ---- Figure ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

# --- Barplot groupé des logits (image 0) ---
ax = axes[0]
num_classes_plot = cifar_logits["learned"].shape[1]
x = torch.arange(num_classes_plot).float().numpy()
bar_width = 0.25

for idx, (name, color) in enumerate(zip(pe_names, pe_colors)):
    vals = cifar_logits[name][0].detach().numpy()
    offsets = x + (idx - 1) * bar_width
    ax.bar(offsets, vals, width=bar_width, color=color, label=name, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(CIFAR10_CLASSES, rotation=30, ha="right", fontsize=9)
ax.set_title("Logits par classe pour l'image 0\n(même image, trois stratégies PE)", fontsize=11)
ax.set_ylabel("Valeur du logit")
ax.legend(title="PE", fontsize=9)
ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")

# --- PCA scatter des tokens CLS ---
ax = axes[1]

for idx, (name, color) in enumerate(zip(pe_names, pe_colors)):
    proj = pca_projections[name]
    for class_idx in range(num_cifar_classes):
        mask = labels_np == class_idx
        if mask.sum() == 0:
            continue
        label = CIFAR10_CLASSES[class_idx] if idx == 0 else None
        ax.scatter(
            proj[mask, 0],
            proj[mask, 1],
            color=cmap(class_idx),
            marker=["o", "s", "^"][idx],
            alpha=0.75,
            s=55,
            label=label,
        )

from matplotlib.lines import Line2D
strategy_handles = [
    Line2D([0], [0], marker=m, color="gray", linestyle="None", markersize=8, label=n)
    for m, n in zip(["o", "s", "^"], pe_names)
]
class_handles = [
    Line2D([0], [0], marker="o", color=cmap(i), linestyle="None", markersize=8, label=CIFAR10_CLASSES[i])
    for i in range(num_cifar_classes)
]

leg1 = ax.legend(handles=strategy_handles, title="Stratégie PE", loc="upper left", fontsize=8)
ax.add_artist(leg1)
ax.legend(handles=class_handles, title="Classe CIFAR-10", loc="lower right", fontsize=7, ncol=2)

ax.set_title("PCA des tokens CLS (batch CIFAR-10)\n(chaque stratégie PE = marqueur différent)", fontsize=11)
ax.set_xlabel("Composante principale 1")
ax.set_ylabel("Composante principale 2")

plt.suptitle(
    "Divergence des représentations selon la stratégie positionnelle (initialisation aléatoire)",
    fontsize=12,
)
plt.show()

---
## 7. Conclusion et perspectives

### Ce que nous avons vu

| Question | Réponse |
|---|---|
| *Pourquoi encoder la position ?* | L'attention est invariante par permutation ; sans PE, un ViT est aveugle à la structure spatiale de l'image. |
| *Absolu appris* | Flexible et simple, mais fixé à la longueur d'entraînement. |
| *Sinusoïdal fixe* | Gratuit en paramètres, avec un biais de localité intégré et une extrapolation partielle. |
| *RoPE* | Rotation de $q$ et $k$, encode la position **relative**, préserve la norme, sans paramètre. |

### Dans ce projet

- **Classification supervisée** (approche classification dans `training_methods.ipynb`) : `position_embedding="learned"`, conforme à ViT original.
- **MAE** (approche MAE dans `training_methods.ipynb`) : `position_embedding="cosine"`, conforme au papier MAE (He et al. 2022), qui utilise le sinusoïdal 2D.
- **DINO** (approche DINO dans `training_methods.ipynb`) : les deux fonctionnent ; `"cosine"` est plus courant en self-supervised learning vision.

### Perspectives

- **RoPE 2D** : étend RoPE aux axes $(x, y)$ séparément (EVA-02, InternViT). La séquence de patches est naturellement 2D, une rotation planaire par axe est plus naturelle qu'une rotation 1D.
- **ALiBi** (Press et al. 2022) : ajoute un biais négatif proportionnel à la distance dans les scores d'attention, simplifie encore en évitant tout vecteur positionnel.
- **Relative position bias** (Shaw et al. 2018, Swin Transformer) : ajoute un biais scalaire appris à chaque paire $(i, j)$ selon leur distance relative, utilisé dans Swin et DeiT-III.
- En NLP, **RoPE est devenu la norme** pour les LLMs récents (LLaMA, Mistral, Qwen). En vision, l'absolu appris reste dominant pour les ViT classiques, mais RoPE 2D gagne du terrain.

---

> *Ce notebook fait partie du projet pédagogique **ViT from scratch**.*  
> *Suite : `vit_building_blocks.ipynb` pour l'anatomie complète du mécanisme d'attention multi-têtes.*